In [1]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import HistGradientBoostingRegressor
import joblib

ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
DATA_RAW = ROOT / "data_raw"
DATA_PROCESSED = ROOT / "data_processed"
REPORTS = ROOT / "reports"
MODELS = ROOT / "models"

DATA_PROCESSED.mkdir(exist_ok=True)
REPORTS.mkdir(exist_ok=True)
MODELS.mkdir(exist_ok=True)

print("ROOT:", ROOT)

ROOT: /Users/jayadeep/GenAI-Weather-Based-Store-Analytics


In [2]:
# =========================
# CLEAN BUILD: perf + weather -> m (guaranteed store_id column)
# =========================
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
DATA_RAW = ROOT / "data_raw"
DP = ROOT / "data_processed"

# ---------- 1) Load performance (true store_id lives here) ----------
perf = pd.read_csv(DATA_RAW / "store_performance_2018to2022.csv")
perf["invoice_date"] = pd.to_datetime(perf["invoice_date"]).dt.normalize()
perf["store_id"] = pd.to_numeric(perf["store_id"], errors="coerce").astype(int)

# enforce unique daily keys
dup = perf.duplicated(["store_id", "invoice_date"]).sum()
print("perf duplicate keys:", dup)
if dup > 0:
    perf = (perf.groupby(["store_id","invoice_date"], as_index=False)
              .agg({"invoice_count":"sum","oc_count":"sum","fleet_oc_count":"sum"}))

# ---------- 2) Load weather parquet (FORCE store_id + invoice_date as COLUMNS) ----------
weather = pd.read_parquet(DP / "weather_allstores.parquet")

# If store_id/invoice_date are in the index, pull them out
if "store_id" not in weather.columns or "invoice_date" not in weather.columns:
    weather = weather.reset_index()

# If names got changed, repair them
if "store_id" not in weather.columns:
    for c in ["store_id_x","store_id_y","index","level_0"]:
        if c in weather.columns:
            weather = weather.rename(columns={c:"store_id"})
            break
if "invoice_date" not in weather.columns:
    for c in ["time","date","invoice_date_x","invoice_date_y","level_1"]:
        if c in weather.columns:
            weather = weather.rename(columns={c:"invoice_date"})
            break

# normalize types
weather["invoice_date"] = pd.to_datetime(weather["invoice_date"]).dt.normalize()
weather["store_id"] = pd.to_numeric(weather["store_id"], errors="coerce").astype(int)

# enforce unique weather keys
weather = weather.drop_duplicates(["store_id","invoice_date"])

print("weather cols:", list(weather.columns))
print("weather stores:", weather["store_id"].nunique(), "rows:", len(weather))

# ---------- 3) Merge to build m ----------
m = perf.merge(weather, on=["store_id","invoice_date"], how="inner")
m = m.sort_values(["store_id","invoice_date"]).reset_index(drop=True)

# ---------- 4) Sanity checks ----------
print("m has store_id?", "store_id" in m.columns)
print("m rows:", len(m))
print("m stores (should be ~434):", m["store_id"].nunique())
print("store_id sample:", m["store_id"].head(10).tolist())

# ---------- 5) Derived targets ----------
m["invoice_count"] = m["invoice_count"].clip(lower=0)
m["oc_count"] = m["oc_count"].clip(lower=0)
m["fleet_oc_count"] = m["fleet_oc_count"].clip(lower=0)
m["non_fleet_oc"] = (m["oc_count"] - m["fleet_oc_count"]).clip(lower=0)

# ---------- 6) Calendar columns ----------
m["dow"] = m["invoice_date"].dt.dayofweek
m["month"] = m["invoice_date"].dt.month
m["day_of_year"] = m["invoice_date"].dt.dayofyear
m["year"] = m["invoice_date"].dt.year
m["is_weekend"] = (m["dow"] >= 5).astype(int)

m.head()

perf duplicate keys: 0
weather cols: ['store_id', 'invoice_date', 'tavg', 'tmin', 'tmax', 'prcp', 'wspd', 'snow']
weather stores: 434 rows: 783590
m has store_id? True
m rows: 756701
m stores (should be ~434): 434
store_id sample: [79609, 79609, 79609, 79609, 79609, 79609, 79609, 79609, 79609, 79609]


,invoice_date,store_id,invoice_count,oc_count,fleet_oc_count,tavg,tmin,tmax,prcp,wspd,snow,non_fleet_oc,dow,month,day_of_year,year,is_weekend
0,2018-01-02,79609,47,39,2,-13.8,-19.3,-6.6,0.0,5.4,0.0,37,1,1,2,2018,0
1,2018-01-03,79609,46,40,8,-9.3,-15.5,-0.5,0.0,13.7,0.0,32,2,1,3,2018,0
2,2018-01-04,79609,43,38,6,-7.9,-13.2,-4.9,0.0,16.9,0.0,32,3,1,4,2018,0
3,2018-01-05,79609,45,30,2,-11.1,-14.9,-7.1,0.0,9.4,0.0,28,4,1,5,2018,0
4,2018-01-06,79609,39,32,0,-12.3,-18.8,-5.5,0.0,4.0,0.0,32,5,1,6,2018,1


In [3]:
# ---- Weather-derived features (buckets + severity) ----
m["rain_mm"] = m["prcp"].fillna(0.0)
m["snow_cm"] = m["snow"].fillna(0.0)

m["tmin_c"] = m["tmin"]
m["tmax_c"] = m["tmax"]

def rain_bucket(x):
    if x <= 0: return "rain_0"
    if x < 5:  return "rain_light"
    if x < 15: return "rain_med"
    return "rain_heavy"

def snow_bucket(x):
    if x <= 0: return "snow_0"
    if x < 2:  return "snow_light"
    if x < 5:  return "snow_med"
    return "snow_heavy"

def heat_bucket(tmax):
    if pd.isna(tmax): return "heat_na"
    if tmax < 30:     return "heat_ok"
    if tmax < 35:     return "heat_hot"
    return "heat_extreme"

def cold_bucket(tmin):
    if pd.isna(tmin): return "cold_na"
    if tmin <= -10:   return "cold_vcold"
    if tmin <= -5:    return "cold_cold"
    if tmin <= 0:     return "cold_freezing"
    return "cold_ok"

m["rain_bucket"] = m["rain_mm"].apply(rain_bucket)
m["snow_bucket"] = m["snow_cm"].apply(snow_bucket)
m["heat_bucket"] = m["tmax_c"].apply(heat_bucket)
m["cold_bucket"] = m["tmin_c"].apply(cold_bucket)

m["heavy_rain"] = (m["rain_bucket"] == "rain_heavy").astype(int)
m["heavy_snow"] = (m["snow_bucket"] == "snow_heavy").astype(int)
m["extreme_heat"] = (m["heat_bucket"] == "heat_extreme").astype(int)
m["extreme_cold"] = (m["cold_bucket"] == "cold_vcold").astype(int)
m["freezing"] = m["tmin_c"].fillna(99).le(0).astype(int)

def severity(row):
    if row["heavy_snow"] == 1: return "sev_snow_heavy"
    if row["heavy_rain"] == 1: return "sev_rain_heavy"
    if row["extreme_heat"] == 1: return "sev_heat_extreme"
    if row["extreme_cold"] == 1: return "sev_cold_extreme"
    if row["freezing"] == 1: return "sev_freezing"
    return "sev_normal"

m["severity"] = m.apply(severity, axis=1)

m["temp_range"] = (m["tmax_c"] - m["tmin_c"]).fillna(0)
m["heavy_rain_weekend"] = m["heavy_rain"] * m["is_weekend"]
m["heavy_snow_weekend"] = m["heavy_snow"] * m["is_weekend"]
m["extreme_heat_weekend"] = m["extreme_heat"] * m["is_weekend"]
m["snow_freezing"] = m["snow_cm"] * m["freezing"]

print("Added buckets/severity OK")

Added buckets/severity OK


In [4]:
store_info = pd.read_csv(DATA_RAW / "store_info.csv")
store_info["store_id"] = pd.to_numeric(store_info["store_id"], errors="coerce").astype(int)

store_meta = store_info[[
    "store_id","bay_count","time_zone_code","closed_day_description",
    "region_id","market_id","area_id","marketing_area_id","store_state"
]].copy()

m = m.merge(store_meta, on="store_id", how="left")
print("Merged store_info OK")

Merged store_info OK


In [5]:
day_map = {"monday":0,"tuesday":1,"wednesday":2,"thursday":3,"friday":4,"saturday":5,"sunday":6}
m["closed_day_description"] = m["closed_day_description"].fillna("").astype(str).str.strip().str.lower()
m["closed_dow"] = m["closed_day_description"].map(day_map)
m["is_closed_day"] = ((m["closed_dow"].notna()) & (m["dow"] == m["closed_dow"])).astype(int)

m["bay_count"] = pd.to_numeric(m["bay_count"], errors="coerce")
m["bay_count"] = m["bay_count"].fillna(m["bay_count"].median())
m["bay_count_log"] = np.log1p(m["bay_count"])

m["is_peak_day"] = m["dow"].isin([3,4,5]).astype(int)
m["capacity_pressure"] = m["is_peak_day"] * (1.0 / m["bay_count"])

print("capacity_pressure + is_closed_day OK")

capacity_pressure + is_closed_day OK


In [6]:
m["heavy_rain_capacity"] = m["heavy_rain"] * m["capacity_pressure"]
m["heavy_snow_capacity"] = m["heavy_snow"] * m["capacity_pressure"]
m["freezing_capacity"]   = m["freezing"]   * m["capacity_pressure"]
m["extreme_heat_capacity"] = m["extreme_heat"] * m["capacity_pressure"]
m["extreme_cold_capacity"] = m["extreme_cold"] * m["capacity_pressure"]
print("capacity interactions OK")

capacity interactions OK


In [7]:
# ===== Fast lag/roll (NO groupby.apply) =====
m = m.sort_values(["store_id","invoice_date"]).reset_index(drop=True)
g = m.groupby("store_id", sort=False)

for tgt in [c for c in ["invoice_count","oc_count","non_fleet_oc","fleet_oc_count"] if c in m.columns]:
    m[f"{tgt}_lag_1"]  = g[tgt].shift(1)
    m[f"{tgt}_lag_7"]  = g[tgt].shift(7)
    m[f"{tgt}_lag_14"] = g[tgt].shift(14)

    m[f"{tgt}_roll7_mean"]  = g[tgt].shift(1).rolling(7,  min_periods=3).mean().reset_index(level=0, drop=True)
    m[f"{tgt}_roll28_mean"] = g[tgt].shift(1).rolling(28, min_periods=10).mean().reset_index(level=0, drop=True)

for col in [c for c in ["rain_mm","snow_cm","tmin_c","tmax_c","wspd","tavg"] if c in m.columns]:
    m[f"{col}_lag_1"] = g[col].shift(1)
    m[f"{col}_roll3_mean"] = g[col].shift(1).rolling(3, min_periods=2).mean().reset_index(level=0, drop=True)
    m[f"{col}_roll7_mean"] = g[col].shift(1).rolling(7, min_periods=3).mean().reset_index(level=0, drop=True)

print("Lag/Roll cols created:", len([c for c in m.columns if "lag_" in c or "roll" in c]))

Lag/Roll cols created: 38


In [8]:
SPLIT_YEAR = 2022
train_df = m[m["year"] < SPLIT_YEAR].copy()
valid_df = m[m["year"] == SPLIT_YEAR].copy()

print("train rows:", len(train_df), "| valid rows:", len(valid_df))
print("Leakage check:", train_df["invoice_date"].max() < valid_df["invoice_date"].min())

train rows: 603834 | valid rows: 152867
Leakage check: True


In [9]:
# Re-define cat_cols + num_cols (run once after rebuilding m + adding store_meta + buckets)
cat_cols = [
    "store_id",
    "rain_bucket","snow_bucket","heat_bucket","cold_bucket","severity",
    "market_id","store_state","time_zone_code",
    "area_id","marketing_area_id"
]

num_cols = [
    "dow","month","day_of_year","is_weekend",
    "tmin_c","tmax_c","tavg","rain_mm","snow_cm","wspd","temp_range",
    "heavy_rain","heavy_snow","extreme_heat","extreme_cold","freezing",
    "heavy_rain_weekend","heavy_snow_weekend","extreme_heat_weekend","snow_freezing",
    "bay_count","bay_count_log","capacity_pressure","is_closed_day",
    "heavy_rain_capacity","heavy_snow_capacity","freezing_capacity",
    "extreme_heat_capacity","extreme_cold_capacity",
]

# Keep only columns that actually exist (prevents crashes)
cat_cols = [c for c in cat_cols if c in train_df.columns]
num_cols = [c for c in num_cols if c in train_df.columns]

print("cat_cols:", cat_cols)
print("num_cols count:", len(num_cols))

cat_cols: ['store_id', 'rain_bucket', 'snow_bucket', 'heat_bucket', 'cold_bucket', 'severity', 'market_id', 'store_state', 'time_zone_code', 'area_id', 'marketing_area_id']
num_cols count: 29


In [10]:
lag_roll_cols = [c for c in train_df.columns if ("lag_" in c or "roll" in c)]
feature_cols = [c for c in (cat_cols + num_cols + lag_roll_cols) if c in train_df.columns]
print("Lag/Roll used:", len([c for c in feature_cols if "lag_" in c or "roll" in c]))
print("Total features:", len(feature_cols))

Lag/Roll used: 38
Total features: 78


In [11]:
# Fix: ensure store_id is a real column in train_df/valid_df (not index)

for name in ["train_df", "valid_df"]:
    d = locals()[name]
    if "store_id" not in d.columns:
        d = d.reset_index()

    # if reset_index created "index" column instead of store_id
    if "store_id" not in d.columns and "index" in d.columns:
        d = d.rename(columns={"index": "store_id"})

    d["store_id"] = pd.to_numeric(d["store_id"], errors="coerce").astype(int)
    locals()[name] = d

print("train_df has store_id?", "store_id" in train_df.columns, "| nunique:", train_df["store_id"].nunique())
print("valid_df has store_id?", "store_id" in valid_df.columns, "| nunique:", valid_df["store_id"].nunique())

train_df has store_id? True | nunique: 431
valid_df has store_id? True | nunique: 434


In [12]:
# categorical + numeric features
cat_cols = [
    "store_id",
    "rain_bucket","snow_bucket","heat_bucket","cold_bucket","severity",
    # add region grouping (treat IDs as categorical)
    "market_id",
    # optional if you want more grouping:
    # "area_id","region_id","marketing_area_id",
    "store_state",
    "time_zone_code",
]

num_cols = [
    "dow","month","day_of_year","is_weekend",
    "tmin_c","tmax_c","tavg","rain_mm","snow_cm","wspd","temp_range",
    "heavy_rain","heavy_snow","extreme_heat","extreme_cold","freezing",
    "heavy_rain_weekend","heavy_snow_weekend","extreme_heat_weekend","snow_freezing",
    

    # capacity + closure
    "bay_count",
    "bay_count_log",
    "capacity_pressure",   # (make sure you used this name)
    "is_closed_day",
]
# add capacity × weather interaction features if they exist
for c in ["heavy_rain_capacity","heavy_snow_capacity","freezing_capacity",
          "extreme_heat_capacity","extreme_cold_capacity"]:
    if c in train_df.columns and c not in num_cols:
        num_cols.append(c)

# add lag/roll features for key signals (include for all three targets)
lag_roll_cols = [c for c in train_df.columns if (
    c.endswith(("_lag_1","_lag_7","_lag_14","_roll7_mean","_roll28_mean","_roll3_mean"))
    and (("invoice_count" in c) or ("non_fleet_oc" in c) or ("fleet_oc_count" in c)
         or ("rain_mm" in c) or ("snow_cm" in c) or ("tmin_c" in c) or ("tmax_c" in c))
)]

feature_cols = [c for c in (cat_cols + num_cols + lag_roll_cols) if c in train_df.columns]

print("Total features:", len(feature_cols))
print("Missing from train_df (cat):", [c for c in cat_cols if c not in train_df.columns])
print("Missing from train_df (num):", [c for c in num_cols if c not in train_df.columns])
print("Example features:", feature_cols[:30])

Total features: 65
Missing from train_df (cat): []
Missing from train_df (num): []
Example features: ['store_id', 'rain_bucket', 'snow_bucket', 'heat_bucket', 'cold_bucket', 'severity', 'market_id', 'store_state', 'time_zone_code', 'dow', 'month', 'day_of_year', 'is_weekend', 'tmin_c', 'tmax_c', 'tavg', 'rain_mm', 'snow_cm', 'wspd', 'temp_range', 'heavy_rain', 'heavy_snow', 'extreme_heat', 'extreme_cold', 'freezing', 'heavy_rain_weekend', 'heavy_snow_weekend', 'extreme_heat_weekend', 'snow_freezing', 'bay_count']


In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

def make_preprocess(cat_cols, feature_cols):
    cat_cols_use = [c for c in cat_cols if c in feature_cols]
    num_cols_use = [c for c in feature_cols if c not in cat_cols_use]

    preprocess = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols_use),
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_cols_use),
    ])
    return preprocess, cat_cols_use, num_cols_use

In [14]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingRegressor
import numpy as np

# Add region/area clustering IDs (categorical)
for c in ["market_id", "area_id", "marketing_area_id"]:
    if c in train_df.columns and c not in cat_cols:
        cat_cols.append(c)

# rebuild feature_cols so they’re actually used
feature_cols = [c for c in (cat_cols + num_cols + lag_roll_cols) if c in train_df.columns]

# rebuild preprocess and retrain
preprocess, cat_used, num_used = make_preprocess(cat_cols, feature_cols)
print("Now using:", [c for c in ["market_id","area_id","marketing_area_id"] if c in cat_used])
def make_preprocess(cat_cols, feature_cols):
    cat_cols_use = [c for c in cat_cols if c in feature_cols]
    num_cols_use = [c for c in feature_cols if c not in cat_cols_use]

    preprocess = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols_use),
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_cols_use),
    ])
    return preprocess, cat_cols_use, num_cols_use

def train_hgb(preprocess, X_train, y_train):
    hgb = HistGradientBoostingRegressor(
        random_state=42,
        learning_rate=0.05,
        max_depth=6,
        max_iter=800
    )
    pipe = Pipeline([
        ("prep", preprocess),
        ("model", hgb)
    ])
    pipe.fit(X_train, y_train)
    return pipe

def safe_mape(y_true, y_pred, min_true=1.0):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    mask = y_true >= min_true
    if mask.sum() == 0:
        return float("nan")
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)

Now using: ['market_id', 'area_id', 'marketing_area_id']


In [19]:
from sklearn.metrics import mean_absolute_error
import numpy as np

preprocess, cat_used, num_used = make_preprocess(cat_cols, feature_cols)

X_train = train_df[feature_cols].copy()
X_valid = valid_df[feature_cols].copy()

targets = {
    "invoice_count": "invoice_count",
    "non_fleet_oc": "non_fleet_oc",
    "fleet_oc_count": "fleet_oc_count"
}

models = {}
metrics = {}

# Ensure non_fleet_oc exists in train_df and valid_df (fixes KeyError)

for d in [train_df, valid_df]:
    if "non_fleet_oc" not in d.columns:
        d["fleet_oc_count"] = d["fleet_oc_count"].clip(lower=0)
        d["oc_count"] = d["oc_count"].clip(lower=0)
        d["non_fleet_oc"] = (d["oc_count"] - d["fleet_oc_count"]).clip(lower=0)

print("train_df has non_fleet_oc?", "non_fleet_oc" in train_df.columns)
print("valid_df has non_fleet_oc?", "non_fleet_oc" in valid_df.columns)

for name, tgt in targets.items():
    y_train = train_df[tgt].astype(float).values
    y_valid = valid_df[tgt].astype(float).values

    
    pipe = train_hgb(preprocess, X_train, y_train)

    pred = np.clip(pipe.predict(X_valid), 0, None)

    mae = mean_absolute_error(y_valid, pred)
    smape = safe_mape(y_valid, pred)

    models[name] = pipe
    metrics[name] = {"mae": float(mae), "safe_mape": float(smape)}

    print(f"{name} -> MAE: {mae:.2f} | safeMAPE: {smape:.2f}%")

train_df has non_fleet_oc? True
valid_df has non_fleet_oc? True
invoice_count -> MAE: 6.38 | safeMAPE: 15.85%
non_fleet_oc -> MAE: 5.53 | safeMAPE: 17.26%
fleet_oc_count -> MAE: 1.37 | safeMAPE: 52.02%


In [20]:
# --- FIX: build valid_out safely even if store_id/invoice_date are in the index ---

import numpy as np
import pandas as pd

# 0) Ensure keys are columns (not index)
if "store_id" not in valid_df.columns or "invoice_date" not in valid_df.columns:
    valid_df = valid_df.reset_index()

# If store_id is still missing, it was probably reset into a generic "index" column
if "store_id" not in valid_df.columns and "index" in valid_df.columns:
    valid_df = valid_df.rename(columns={"index": "store_id"})

# Safety: make invoice_date proper datetime
valid_df["invoice_date"] = pd.to_datetime(valid_df["invoice_date"]).dt.normalize()

# 1) Predictions
pred_invoice   = np.clip(models["invoice_count"].predict(X_valid), 0, None)
pred_nonfleet  = np.clip(models["non_fleet_oc"].predict(X_valid), 0, None)
pred_fleet     = np.clip(models["fleet_oc_count"].predict(X_valid), 0, None)

# 2) Derived + enforce consistency
pred_oc = pred_nonfleet + pred_fleet
pred_oc = np.minimum(pred_oc, pred_invoice)          # OC <= invoices
pred_fleet = np.minimum(pred_fleet, pred_oc)         # fleet <= total OC
pred_fleet = np.maximum(pred_fleet, 0)               # non-negative
pred_nonfleet = np.maximum(pred_oc - pred_fleet, 0)  # remainder

# 3) Build output table (only select columns that exist)
base_cols = ["store_id","invoice_date","invoice_count","oc_count","fleet_oc_count","non_fleet_oc","severity"]
base_cols = [c for c in base_cols if c in valid_df.columns]

valid_out = valid_df[base_cols].copy()
valid_out["pred_invoice"] = pred_invoice
valid_out["pred_non_fleet_oc"] = pred_nonfleet
valid_out["pred_fleet_oc"] = pred_fleet
valid_out["pred_oc_total"] = pred_oc

# 4) Errors (if actuals exist)
if "invoice_count" in valid_out.columns:
    valid_out["abs_err_invoice"] = (valid_out["invoice_count"].astype(float) - valid_out["pred_invoice"]).abs()
if "oc_count" in valid_out.columns:
    valid_out["abs_err_oc"] = (valid_out["oc_count"].astype(float) - valid_out["pred_oc_total"]).abs()

print("valid_out columns:", list(valid_out.columns))
valid_out.head()

valid_out columns: ['store_id', 'invoice_date', 'invoice_count', 'oc_count', 'fleet_oc_count', 'non_fleet_oc', 'severity', 'pred_invoice', 'pred_non_fleet_oc', 'pred_fleet_oc', 'pred_oc_total', 'abs_err_invoice', 'abs_err_oc']


,store_id,invoice_date,invoice_count,oc_count,fleet_oc_count,non_fleet_oc,severity,pred_invoice,pred_non_fleet_oc,pred_fleet_oc,pred_oc_total,abs_err_invoice,abs_err_oc
1432,79609,2022-01-02,37,31,3,28,sev_freezing,34.352235,27.230330,0.900923,28.131253,2.647765,2.868747
1433,79609,2022-01-03,60,55,7,48,sev_freezing,58.877573,43.321272,5.126085,48.447357,1.122427,6.552643
1434,79609,2022-01-04,62,50,5,45,sev_freezing,52.989316,40.701887,5.104282,45.806169,9.010684,4.193831
1435,79609,2022-01-05,52,44,4,40,sev_freezing,52.737211,37.105519,4.691877,41.797396,0.737211,2.202604
1436,79609,2022-01-06,16,13,1,12,sev_freezing,39.070512,26.592101,4.249326,30.841427,23.070512,17.841427


In [21]:
import numpy as np

def build_intervals(y_true, y_pred, severity_series, min_bucket=200):
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    resid = y_true - y_pred

    # Global interval
    q_lo_g = np.nanpercentile(resid, 2.5)
    q_hi_g = np.nanpercentile(resid, 97.5)
    if q_hi_g < q_lo_g:
        q_lo_g, q_hi_g = q_hi_g, q_lo_g

    sev = severity_series.astype(str).values
    sev_bounds = {}

    for s in np.unique(sev):
        r = resid[sev == s]
        r = r[~np.isnan(r)]
        if len(r) < min_bucket:
            sev_bounds[s] = (q_lo_g, q_hi_g)
        else:
            lo = np.nanpercentile(r, 2.5)
            hi = np.nanpercentile(r, 97.5)
            if hi < lo:
                lo, hi = hi, lo
            sev_bounds[s] = (lo, hi)

    return (q_lo_g, q_hi_g), sev_bounds

# Build intervals for OC total
y_true_oc = valid_out["oc_count"].astype(float).values
y_pred_oc = valid_out["pred_oc_total"].astype(float).values

(global_lo, global_hi), sev_bounds = build_intervals(y_true_oc, y_pred_oc, valid_out["severity"])

# Apply bounds
lows, highs = [], []
for p, s in zip(y_pred_oc, valid_out["severity"].astype(str).values):
    lo, hi = sev_bounds.get(s, (global_lo, global_hi))
    lo_val = max(p + lo, 0)
    hi_val = max(p + hi, lo_val)   # ensure hi >= lo and >=0
    lows.append(lo_val)
    highs.append(hi_val)

valid_out["oc_lower_95"] = lows
valid_out["oc_upper_95"] = highs

inside = ((valid_out["oc_count"] >= valid_out["oc_lower_95"]) &
          (valid_out["oc_count"] <= valid_out["oc_upper_95"])).mean()

print("OC 95% interval coverage:", round(float(inside) * 100, 2), "%")
valid_out[["store_id","invoice_date","oc_count","pred_oc_total","oc_lower_95","oc_upper_95","severity"]].head()

OC 95% interval coverage: 95.0 %


,store_id,invoice_date,oc_count,pred_oc_total,oc_lower_95,oc_upper_95,severity
1432,79609,2022-01-02,31,28.131253,13.174609,43.718089,sev_freezing
1433,79609,2022-01-03,55,48.447357,33.490713,64.034193,sev_freezing
1434,79609,2022-01-04,50,45.806169,30.849525,61.393005,sev_freezing
1435,79609,2022-01-05,44,41.797396,26.840752,57.384232,sev_freezing
1436,79609,2022-01-06,13,30.841427,15.884783,46.428263,sev_freezing


In [22]:
# Fix: ensure store_id + invoice_date are real columns (not index) for heuristic
for name, d in [("train_df", train_df), ("valid_df", valid_df)]:
    if "store_id" not in d.columns or "invoice_date" not in d.columns:
        d2 = d.reset_index()
        # if store_id got dumped into a generic column
        if "store_id" not in d2.columns and "index" in d2.columns:
            d2 = d2.rename(columns={"index": "store_id"})
        if name == "train_df":
            train_df = d2
        else:
            valid_df = d2

print("train_df cols has store_id?", "store_id" in train_df.columns)
print("valid_df cols has store_id?", "store_id" in valid_df.columns)

train_df cols has store_id? True
valid_df cols has store_id? True


In [23]:
# Log-additive heuristic (more stable than multiplying 4 bucket multipliers)

import numpy as np
import json
from sklearn.metrics import mean_absolute_error

train = train_df.copy()
valid = valid_df.copy()

# 1) Baseline store+dow
base_tbl = (train.groupby(["store_id","dow"])["oc_count"].mean().rename("base_oc").reset_index())
train = train.merge(base_tbl, on=["store_id","dow"], how="left")
valid = valid.merge(base_tbl, on=["store_id","dow"], how="left")

# fallback baselines
store_mean = train.groupby("store_id")["oc_count"].mean().rename("store_mean_oc")
global_mean = float(train["oc_count"].mean())
train = train.merge(store_mean, on="store_id", how="left")
valid = valid.merge(store_mean, on="store_id", how="left")
train["base_oc"] = train["base_oc"].fillna(train["store_mean_oc"]).fillna(global_mean)
valid["base_oc"] = valid["base_oc"].fillna(valid["store_mean_oc"]).fillna(global_mean)

# 2) Trend ratio (use oc_count rolls if available)
def pick_trend_cols(d):
    candidates = [
        ("oc_count_roll7_mean", "oc_count_roll28_mean"),
        ("non_fleet_oc_roll7_mean", "non_fleet_oc_roll28_mean"),
        ("invoice_count_roll7_mean", "invoice_count_roll28_mean"),
    ]
    for a, b in candidates:
        if a in d.columns and b in d.columns:
            return a, b
    return None, None

trend_used = None
for d in [train, valid]:
    c7, c28 = pick_trend_cols(d)
    if c7 is None:
        d["trend_ratio"] = 1.0
    else:
        trend_used = f"{c7}/{c28}"
        d["trend_ratio"] = (
            (d[c7] / d[c28]).replace([np.inf,-np.inf], np.nan).fillna(1.0).clip(0.7, 1.3)
        )
    d["base_trend"] = (d["base_oc"] * d["trend_ratio"]).clip(lower=1e-6)

print("Trend ratio using:", trend_used if trend_used else "fallback=1.0")

# 3) Learn log-effects (additive, prevents runaway scaling)
# log_ratio = log(actual/base_trend)
train["log_ratio"] = np.log((train["oc_count"].astype(float) + 1e-6) / train["base_trend"].astype(float))
train["log_ratio"] = train["log_ratio"].replace([np.inf,-np.inf], np.nan)

def med_log_map(df, col):
    return df.dropna(subset=["log_ratio"]).groupby(col)["log_ratio"].median().to_dict()

rain_log = med_log_map(train, "rain_bucket")
snow_log = med_log_map(train, "snow_bucket")
heat_log = med_log_map(train, "heat_bucket")
cold_log = med_log_map(train, "cold_bucket")

def get_log(mp, key):
    v = mp.get(key, 0.0)  # 0 means "no effect"
    try:
        v = float(v)
    except Exception:
        return 0.0
    return v if np.isfinite(v) else 0.0

valid["log_rain"] = valid["rain_bucket"].apply(lambda x: get_log(rain_log, x))
valid["log_snow"] = valid["snow_bucket"].apply(lambda x: get_log(snow_log, x))
valid["log_heat"] = valid["heat_bucket"].apply(lambda x: get_log(heat_log, x))
valid["log_cold"] = valid["cold_bucket"].apply(lambda x: get_log(cold_log, x))

# Total log adjustment (additive)
valid["log_adj"] = valid["log_rain"] + valid["log_snow"] + valid["log_heat"] + valid["log_cold"]

# Convert back: pred = base_trend * exp(log_adj)
valid["oc_pred_heuristic_logadd"] = (valid["base_trend"] * np.exp(valid["log_adj"])).clip(lower=0)

# 4) Metrics
h_mae = mean_absolute_error(valid["oc_count"].values, valid["oc_pred_heuristic_logadd"].values)
h_smape = safe_mape(valid["oc_count"].values, valid["oc_pred_heuristic_logadd"].values)

print("Heuristic (log-additive) OC -> MAE:", round(float(h_mae),2), "| safeMAPE:", round(float(h_smape),2), "%")

# 5) Save artifacts
heur = {
    "target": "oc_count",
    "trend_ratio_used": trend_used if trend_used else "none (1.0)",
    "rain_log": rain_log,
    "snow_log": snow_log,
    "heat_log": heat_log,
    "cold_log": cold_log,
    "note": "log-additive heuristic: pred = base_trend * exp(sum(median log effects))"
}
(REPORTS / "heuristic_multipliers_oc_logadd.json").write_text(json.dumps(heur, indent=2))
print("Saved:", REPORTS / "heuristic_multipliers_oc_logadd.json")

out = valid[[
    "store_id","invoice_date","oc_count","base_oc","trend_ratio","base_trend",
    "rain_bucket","snow_bucket","heat_bucket","cold_bucket",
    "log_rain","log_snow","log_heat","log_cold","log_adj",
    "oc_pred_heuristic_logadd"
]].copy()
out.to_csv(DATA_PROCESSED / "predictions_heuristic_oc_2022_logadd.csv", index=False)
print("Saved:", DATA_PROCESSED / "predictions_heuristic_oc_2022_logadd.csv")

Trend ratio using: oc_count_roll7_mean/oc_count_roll28_mean
Heuristic (log-additive) OC -> MAE: 8.37 | safeMAPE: 22.05 %
Saved: /Users/jayadeep/GenAI-Weather-Based-Store-Analytics/reports/heuristic_multipliers_oc_logadd.json
Saved: /Users/jayadeep/GenAI-Weather-Based-Store-Analytics/data_processed/predictions_heuristic_oc_2022_logadd.csv


In [35]:
# =========================
# Heuristic v2 vs HGB comparison table (2022)
# Makes Heuristic task "100% done"
# =========================
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import mean_absolute_error

ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
DP = ROOT / "data_processed"

# --- EDIT THESE IF YOUR FILENAMES ARE DIFFERENT ---
HGB_PATH  = DP / "predictions_hgb_hierarchical_2022.csv"      # must contain oc_count + pred_oc_total (or similar)
HEUR_PATH = DP / "predictions_heuristic_oc_2022.csv"          # must contain oc_actual + oc_pred_heuristic_v2
OUT_PATH  = DP / "comparison_heuristic_vs_hgb_oc_2022.csv"

def safe_mape(y_true, y_pred, min_true=1.0):
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    mask = y_true >= min_true
    if mask.sum() == 0:
        return float("nan")
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)

# --- Load HGB predictions ---
hgb = pd.read_csv(HGB_PATH)
hgb["invoice_date"] = pd.to_datetime(hgb["invoice_date"]).dt.normalize()

# Find actual and pred columns for OC in HGB file
actual_candidates = [c for c in ["oc_count","actual_oc","y_true_oc","oc_actual"] if c in hgb.columns]
pred_candidates   = [c for c in ["pred_oc_total","pred_oc","y_pred_oc","oc_pred"] if c in hgb.columns]

if not actual_candidates or not pred_candidates:
    raise ValueError(f"HGB file missing OC columns. Columns: {list(hgb.columns)}")

hgb_actual = actual_candidates[0]
hgb_pred   = pred_candidates[0]

hgb_keep = hgb[["store_id","invoice_date", hgb_actual, hgb_pred]].copy()
hgb_keep = hgb_keep.rename(columns={hgb_actual:"oc_actual", hgb_pred:"oc_pred_hgb"})

# --- Load Heuristic predictions ---
heur = pd.read_csv(HEUR_PATH)
heur["invoice_date"] = pd.to_datetime(heur["invoice_date"]).dt.normalize()

# Find heuristic actual/pred columns
heur_actual_candidates = [c for c in ["oc_actual","oc_count"] if c in heur.columns]
heur_pred_candidates   = [c for c in ["oc_pred_heuristic_v2","oc_pred_heuristic","oc_pred_heuristic_logadd","oc_pred"] if c in heur.columns]

if not heur_actual_candidates or not heur_pred_candidates:
    raise ValueError(f"Heuristic file missing columns. Columns: {list(heur.columns)}")

heur_actual = heur_actual_candidates[0]
heur_pred   = heur_pred_candidates[0]

heur_keep = heur[["store_id","invoice_date", heur_actual, heur_pred]].copy()
heur_keep = heur_keep.rename(columns={heur_actual:"oc_actual_heur", heur_pred:"oc_pred_heuristic"})

# --- Merge for fair comparison on same rows ---
cmp = hgb_keep.merge(heur_keep, on=["store_id","invoice_date"], how="inner")

# Use a single actual column (prefer HGB actual)
cmp["oc_actual_final"] = cmp["oc_actual"].astype(float)

# Metrics
mae_hgb  = mean_absolute_error(cmp["oc_actual_final"], cmp["oc_pred_hgb"])
mape_hgb = safe_mape(cmp["oc_actual_final"], cmp["oc_pred_hgb"])

mae_heur  = mean_absolute_error(cmp["oc_actual_final"], cmp["oc_pred_heuristic"])
mape_heur = safe_mape(cmp["oc_actual_final"], cmp["oc_pred_heuristic"])

# Improvement (positive = heuristic better)
mae_improve  = mae_hgb - mae_heur
mape_improve = mape_hgb - mape_heur

summary = pd.DataFrame([
    {"model":"HGB (hierarchical OC)", "rows":len(cmp), "MAE":mae_hgb,  "safeMAPE_%":mape_hgb},
    {"model":"Heuristic v2 (OC)",     "rows":len(cmp), "MAE":mae_heur, "safeMAPE_%":mape_heur},
])

display(summary)

print("\nImprovement (positive means Heuristic better than HGB):")
print("MAE improvement:", round(float(mae_improve), 2))
print("safeMAPE improvement:", round(float(mape_improve), 2), "%")

# Save comparison table (row-level)
cmp_out = cmp[["store_id","invoice_date","oc_actual_final","oc_pred_hgb","oc_pred_heuristic"]].copy()
cmp_out["abs_err_hgb"]  = (cmp_out["oc_actual_final"] - cmp_out["oc_pred_hgb"]).abs()
cmp_out["abs_err_heur"] = (cmp_out["oc_actual_final"] - cmp_out["oc_pred_heuristic"]).abs()
cmp_out.to_csv(OUT_PATH, index=False)

print("\nSaved row-level comparison:", OUT_PATH)
print("Top 10 days where Heuristic beats HGB most (abs_err_hgb - abs_err_heur):")
cmp_out["gain_vs_hgb"] = cmp_out["abs_err_hgb"] - cmp_out["abs_err_heur"]
display(cmp_out.sort_values("gain_vs_hgb", ascending=False).head(10))

,model,rows,MAE,safeMAPE_%
0,HGB (hierarchical OC),58,6.618042,21.24825
1,Heuristic v2 (OC),58,18.073679,43.30779



Improvement (positive means Heuristic better than HGB):
MAE improvement: -11.46
safeMAPE improvement: -22.06 %

Saved row-level comparison: /Users/jayadeep/GenAI-Weather-Based-Store-Analytics/data_processed/comparison_heuristic_vs_hgb_oc_2022.csv
Top 10 days where Heuristic beats HGB most (abs_err_hgb - abs_err_heur):


,store_id,invoice_date,oc_actual_final,oc_pred_hgb,oc_pred_heuristic,abs_err_hgb,abs_err_heur,gain_vs_hgb
1,84831,2022-09-23,40.0,34.914523,40.944647,5.085477,0.944647,4.140830
43,232126,2022-12-26,45.0,56.461489,37.521270,11.461489,7.478730,3.982759
18,99134,2022-10-11,46.0,33.783290,37.530371,12.216710,8.469629,3.747081
52,609430,2022-11-20,37.0,48.158038,45.620694,11.158038,8.620694,2.537344
29,102311,2022-04-06,43.0,49.135030,38.429001,6.135030,4.570999,1.564031
55,612516,2022-10-15,48.0,53.363325,43.919697,5.363325,4.080303,1.283022
7,88323,2022-07-07,40.0,43.897847,37.166368,3.897847,2.833632,1.064215
4,84856,2022-10-18,40.0,43.310769,42.441561,3.310769,2.441561,0.869208
46,232131,2022-12-31,41.0,39.128258,42.292321,1.871742,1.292321,0.579421
39,232122,2022-12-21,61.0,42.641411,42.929855,18.358589,18.070145,0.288444


In [24]:
# Save models
joblib.dump(models["invoice_count"], MODELS / "model_hgb_invoice.joblib")
joblib.dump(models["non_fleet_oc"], MODELS / "model_hgb_non_fleet_oc.joblib")
joblib.dump(models["fleet_oc_count"], MODELS / "model_hgb_fleet_oc.joblib")

# Save predictions (2022 validation)
pred_path = DATA_PROCESSED / "predictions_hgb_hierarchical_2022.csv"
valid_out.to_csv(pred_path, index=False)

# Save metrics + interval coverage
summary = {
    "split": "train=2018-2021, valid=2022",
    "models": metrics,
    "oc_interval_coverage_pct": float(inside)*100,
    "feature_count": len(feature_cols),
    "feature_cols": feature_cols,
}
(REPORTS / "metrics_hgb_hierarchical.json").write_text(json.dumps(summary, indent=2))

print("Saved models to:", MODELS)
print("Saved predictions:", pred_path)
print("Saved metrics:", REPORTS / "metrics_hgb_hierarchical.json")

Saved models to: /Users/jayadeep/GenAI-Weather-Based-Store-Analytics/models
Saved predictions: /Users/jayadeep/GenAI-Weather-Based-Store-Analytics/data_processed/predictions_hgb_hierarchical_2022.csv
Saved metrics: /Users/jayadeep/GenAI-Weather-Based-Store-Analytics/reports/metrics_hgb_hierarchical.json


In [25]:
# OC impact by severity bucket (actual vs predicted + avg delta)
t = valid_out.copy()
t["delta"] = t["oc_count"] - t["pred_oc_total"]
impact = (t.groupby("severity")
            .agg(days=("delta","count"),
                 avg_actual=("oc_count","mean"),
                 avg_pred=("pred_oc_total","mean"),
                 avg_delta=("delta","mean"),
                 mae=("abs_err_oc","mean"))
            .reset_index()
            .sort_values("days", ascending=False))
impact

,severity,days,avg_actual,avg_pred,avg_delta,mae
3,sev_normal,97245,46.713929,46.411626,0.302302,5.433359
1,sev_freezing,26832,45.446631,44.785447,0.661184,5.913162
5,sev_snow_heavy,10520,43.172909,43.588815,-0.415907,6.548633
4,sev_rain_heavy,7911,42.814183,43.097949,-0.283766,6.044869
2,sev_heat_extreme,5972,45.011052,44.910757,0.100294,5.293189
0,sev_cold_extreme,4387,43.158195,43.598538,-0.440344,6.730027


In [26]:
# Turn your severity table into clean % impact bullets vs normal

import pandas as pd

# paste your table here (or set impact = your existing dataframe)
impact = pd.DataFrame([
    {"severity":"sev_normal","days":97245,"avg_actual":46.713929,"avg_pred":46.457841,"avg_delta":0.256088,"mae":5.478188},
    {"severity":"sev_freezing","days":26832,"avg_actual":45.446631,"avg_pred":44.887689,"avg_delta":0.558942,"mae":5.934163},
    {"severity":"sev_snow_heavy","days":10520,"avg_actual":43.172909,"avg_pred":43.752063,"avg_delta":-0.579154,"mae":6.560820},
    {"severity":"sev_rain_heavy","days":7911,"avg_actual":42.814183,"avg_pred":43.104488,"avg_delta":-0.290305,"mae":6.097986},
    {"severity":"sev_heat_extreme","days":5972,"avg_actual":45.011052,"avg_pred":45.043282,"avg_delta":-0.032231,"mae":5.350378},
    {"severity":"sev_cold_extreme","days":4387,"avg_actual":43.158195,"avg_pred":43.841196,"avg_delta":-0.683001,"mae":6.647988},
])

normal = impact.loc[impact["severity"]=="sev_normal","avg_actual"].iloc[0]

impact["delta_vs_normal"] = impact["avg_actual"] - normal
impact["pct_vs_normal"] = (impact["delta_vs_normal"] / normal) * 100

# Pretty labels
labels = {
    "sev_normal":"Normal",
    "sev_freezing":"Freezing (tmin ≤ 0°C)",
    "sev_rain_heavy":"Heavy rain",
    "sev_snow_heavy":"Heavy snow",
    "sev_cold_extreme":"Extreme cold (tmin ≤ -10°C)",
    "sev_heat_extreme":"Extreme heat (tmax ≥ 35°C)",
}
impact["label"] = impact["severity"].map(labels).fillna(impact["severity"])

# Print bullets (skip normal)
rows = impact[impact["severity"]!="sev_normal"].sort_values("pct_vs_normal")

print(f"Baseline (Normal days) avg oil changes: {normal:.1f}\n")

for _, r in rows.iterrows():
    print(
        f"- {r['label']}: avg {r['avg_actual']:.1f} "
        f"({r['delta_vs_normal']:+.1f}, {r['pct_vs_normal']:+.1f}%) "
        f"| days={int(r['days'])}, MAE={r['mae']:.2f}"
    )

Baseline (Normal days) avg oil changes: 46.7

- Heavy rain: avg 42.8 (-3.9, -8.3%) | days=7911, MAE=6.10
- Extreme cold (tmin ≤ -10°C): avg 43.2 (-3.6, -7.6%) | days=4387, MAE=6.65
- Heavy snow: avg 43.2 (-3.5, -7.6%) | days=10520, MAE=6.56
- Extreme heat (tmax ≥ 35°C): avg 45.0 (-1.7, -3.6%) | days=5972, MAE=5.35
- Freezing (tmin ≤ 0°C): avg 45.4 (-1.3, -2.7%) | days=26832, MAE=5.93


“Oil-change demand is most impacted during heavy rain, heavy snow, and extreme cold days, showing roughly a 7–8% reduction versus normal conditions.”

In [34]:
# =========================
# Does weather impact business? (YES/NO) + degrees + 95% confidence
# Requires: a 2022 dataframe with columns: store_id, invoice_date, severity, and KPI column
# Set KPI = "invoice_count" (traffic) or "oc_count" (oil changes)
# =========================
import numpy as np
import pandas as pd

KPI = "invoice_count"  # change to "oc_count" if you want oil-change impact

# Use valid_out if available (preferred), otherwise fall back to valid_df
df = valid_out.copy() if "valid_out" in globals() else valid_df.copy()

# Basic guards
df["invoice_date"] = pd.to_datetime(df["invoice_date"]).dt.normalize()
if "severity" not in df.columns:
    raise ValueError("Missing 'severity'. Create severity buckets first (sev_normal, sev_freezing, etc.).")
if KPI not in df.columns:
    raise ValueError(f"Missing KPI '{KPI}'. Available columns: {list(df.columns)[:30]}")

# Restrict to 2022 for the impact statement
df = df[df["invoice_date"].dt.year == 2022].copy()
df["dow"] = df["invoice_date"].dt.dayofweek
df["month"] = df["invoice_date"].dt.month

# --- Define 'normal day' baseline using ONLY sev_normal days ---
normal = df[df["severity"] == "sev_normal"].copy()
if len(normal) < 1000:
    raise ValueError("Not enough sev_normal rows to define baseline. Check severity logic.")

# Baseline per store + dow + month (best)
base = (normal.groupby(["store_id","dow","month"])[KPI]
        .mean().rename("baseline").reset_index())
df = df.merge(base, on=["store_id","dow","month"], how="left")

# Fallback: store + dow
base2 = (normal.groupby(["store_id","dow"])[KPI]
         .mean().rename("baseline2").reset_index())
df = df.merge(base2, on=["store_id","dow"], how="left")

global_normal = float(normal[KPI].mean())
df["baseline_final"] = df["baseline"].fillna(df["baseline2"]).fillna(global_normal)

# % change vs normal baseline
df["pct_change_vs_normal"] = ((df[KPI] - df["baseline_final"]) / df["baseline_final"]) * 100.0

# --- Bootstrap 95% CI for mean % impact per severity ---
rng = np.random.default_rng(42)

def bootstrap_ci_mean(values, n_boot=2000, alpha=0.05):
    vals = np.asarray(values, float)
    vals = vals[np.isfinite(vals)]
    if len(vals) < 50:
        return (np.nan, np.nan)
    boots = rng.choice(vals, size=(n_boot, len(vals)), replace=True).mean(axis=1)
    lo = float(np.quantile(boots, alpha/2))
    hi = float(np.quantile(boots, 1 - alpha/2))
    return (lo, hi)

def pretty(sev):
    mapping = {
        "sev_normal": "Normal",
        "sev_freezing": "Freezing (tmin ≤ 0°C)",
        "sev_rain_heavy": "Heavy rain",
        "sev_snow_heavy": "Heavy snow",
        "sev_heat_extreme": "Extreme heat (tmax ≥ 35°C)",
        "sev_cold_extreme": "Extreme cold (tmin ≤ -10°C)",
    }
    return mapping.get(sev, sev)

rows = []
for sev in sorted(df["severity"].unique()):
    vals = df.loc[df["severity"] == sev, "pct_change_vs_normal"].values
    mean_pct = float(np.nanmean(vals))
    lo, hi = bootstrap_ci_mean(vals, n_boot=2000, alpha=0.05)
    significant_95 = (np.isfinite(lo) and np.isfinite(hi) and (lo > 0 or hi < 0))
    rows.append({
        "severity": sev,
        "label": pretty(sev),
        "days": int(np.isfinite(vals).sum()),
        "mean_pct_change": mean_pct,
        "ci95_lo": lo,
        "ci95_hi": hi,
        "significant_at_95": bool(significant_95),
    })

impact_ci = pd.DataFrame(rows).sort_values("days", ascending=False)

# --- YES/NO conclusion ---
sig = impact_ci[(impact_ci["severity"] != "sev_normal") & (impact_ci["significant_at_95"])].copy()
overall_yes = len(sig) > 0

print(f"\nFINAL ANSWER (KPI={KPI})")
print("Weather impact on business:", "YES" if overall_yes else "NO",
      "— based on 95% confidence intervals vs normal-day baseline.\n")

print("Degrees of impact vs normal (mean % change, 95% CI):")
for _, r in impact_ci.sort_values("severity").iterrows():
    if r["severity"] == "sev_normal":
        continue
    direction = "more" if r["mean_pct_change"] > 0 else "fewer"
    badge = "" if r["significant_at_95"] else ""
    print(f"- {r['label']}: {abs(r['mean_pct_change']):.1f}% {direction} than normal "
          f"(95% CI: {r['ci95_lo']:.1f}% to {r['ci95_hi']:.1f}%, n={int(r['days'])}){badge}")

display(impact_ci)


FINAL ANSWER (KPI=invoice_count)
Weather impact on business: YES — based on 95% confidence intervals vs normal-day baseline.

Degrees of impact vs normal (mean % change, 95% CI):
- Extreme cold (tmin ≤ -10°C): 5.4% fewer than normal (95% CI: -6.1% to -4.6%, n=4387)
- Freezing (tmin ≤ 0°C): inf% more than normal (95% CI: 1.7% to 2.3%, n=26829)
- Extreme heat (tmax ≥ 35°C): 2.1% more than normal (95% CI: 1.5% to 2.7%, n=5972)
- Heavy rain: 5.1% fewer than normal (95% CI: -5.6% to -4.5%, n=7911)
- Heavy snow: 5.5% fewer than normal (95% CI: -6.0% to -5.0%, n=10520)


,severity,label,days,mean_pct_change,ci95_lo,ci95_hi,significant_at_95
3,sev_normal,Normal,97240,1.337200e-17,-0.084984,0.086802,False
1,sev_freezing,Freezing (tmin ≤ 0°C),26829,inf,1.670390,2.286492,True
5,sev_snow_heavy,Heavy snow,10520,-5.533903e+00,-6.025239,-5.022970,True
4,sev_rain_heavy,Heavy rain,7911,-5.081898e+00,-5.601862,-4.543489,True
2,sev_heat_extreme,Extreme heat (tmax ≥ 35°C),5972,2.100899e+00,1.525046,2.675283,True
0,sev_cold_extreme,Extreme cold (tmin ≤ -10°C),4387,-5.362745e+00,-6.106674,-4.568144,True


In [38]:
from pathlib import Path

ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
DP = ROOT / "data_processed"
DP.mkdir(exist_ok=True)

# impact_ci must exist (from the "YES/NO + 95% CI" cell)
out_path = DP / "weather_impact_ci_2022.csv"
impact_ci.to_csv(out_path, index=False)
print("Saved:", out_path)

Saved: /Users/jayadeep/GenAI-Weather-Based-Store-Analytics/data_processed/weather_impact_ci_2022.csv


In [39]:
from pathlib import Path

ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
REPORTS = ROOT / "reports"
REPORTS.mkdir(exist_ok=True)

# Use whichever one you have:
if "impact" in globals() and isinstance(impact, pd.DataFrame):
    sev_path = REPORTS / "severity_impact_table_2022.csv"
    impact.to_csv(sev_path, index=False)
    print("Saved:", sev_path)
else:
    # fallback: create severity table from valid_out if impact isn't in memory
    t = valid_out.copy()
    if "abs_err_oc" not in t.columns:
        t["abs_err_oc"] = (t["oc_count"] - t["pred_oc_total"]).abs()

    severity_table = (t.groupby("severity")
                        .agg(days=("severity","count"),
                             avg_actual=("oc_count","mean"),
                             avg_pred=("pred_oc_total","mean"),
                             avg_delta=((lambda x: (t.loc[x.index,"oc_count"] - t.loc[x.index,"pred_oc_total"]).mean())),
                             mae=("abs_err_oc","mean"))
                        .reset_index()
                        .sort_values("days", ascending=False))

    sev_path = REPORTS / "severity_impact_table_2022.csv"
    severity_table.to_csv(sev_path, index=False)
    print("Saved:", sev_path)

Saved: /Users/jayadeep/GenAI-Weather-Based-Store-Analytics/reports/severity_impact_table_2022.csv
